# # SageMaker Studio Demo: Pipeline

A training pipeline that runs on SageMaker compute as a DAG.

```
preprocess -> train -> evaluate -> [rmse gate] -> register
```

1. upsert the pipeline defined in `src/pipeline.py`
2. start a run and watch the steps
3. check the metrics
4. find the new version sitting in the registry, pending approval


## 1. Setup

The notebook runs from `notebooks/`, but the pipeline's
`source_dir="src"` and `code="src/..."` paths resolve against the repo
root.

In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

The three values below come from Terraform:

```
terraform -chdir=infra output -raw data_bucket
terraform -chdir=infra output -raw alice_role_arn
terraform -chdir=infra output -raw model_package_group
```

In [ ]:
import boto3

REGION = "ca-central-1"

BUCKET = "mlops-sagemaker-studio-dev-data-3vi8kw"
ROLE = "arn:aws:iam::099139718958:role/mlops-sagemaker-studio-dev-role-alice"
MODEL_PACKAGE_GROUP = "mlops-sagemaker-studio-dev-bike-sharing-rf"

sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)

sm.describe_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP)
print(f"registry group {MODEL_PACKAGE_GROUP} reachable")

## 2. Build and upsert

`build()` returns the Pipeline object; nothing exists in AWS until
`upsert()`. Upsert rather than create -- rerunning this cell is the
normal way to iterate on a definition.

In [ ]:
import sys

# src/ is a directory of entry-point scripts, not a package -- adding an
# __init__.py would ship it to the container too.
sys.path.insert(0, "src")

from pipeline import PIPELINE_NAME, SKLEARN_IMAGE, build

pipeline = build(
    bucket=BUCKET,
    role=ROLE,
    model_package_group=MODEL_PACKAGE_GROUP,
    image=SKLEARN_IMAGE,
    instance_type="ml.m5.large",
    rmse_threshold=150.0,
)

pipeline.upsert(role_arn=ROLE)
print(f"upserted {PIPELINE_NAME}")

The definition is JSON underneath -- what the `aws_sagemaker_pipeline`
resource would have needed hand-written, and what the SDK generated
instead.

In [ ]:
import json

raw = pipeline.definition()
definition = json.loads(raw)

print(f"{len(definition['Steps'])} top-level steps")
for step in definition["Steps"]:
    print(f"  {step['Name']:12} {step['Type']}")

print()
print(f"definition is {len(raw):,} characters of JSON")

## 3. Run it

This is the billed part: three jobs on `ml.m5.large`, a few minutes
each. Instances are provisioned and torn down per step -- nothing keeps
running afterwards, unlike the JupyterLab app.

In [5]:
execution = pipeline.start()

print(execution.arn)

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=7587228;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=7587229;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

arn:aws:sagemaker:ca-central-1:099139718958:pipeline/bike-sharing-rf/execution/k8oram8w1his


In [ ]:
# Roughly 8-12 minutes cold; a rerun that hits the step cache is faster.
execution.wait()

print(execution.describe()["PipelineExecutionStatus"])

In [7]:
for step in execution.list_steps():
    status = step["StepStatus"]
    cached = " (cached)" if step.get("CacheHitResult", {}).get("SourcePipelineExecutionArn") else ""
    print(f"{step['StepName']:12} {status}{cached}")

Register     Succeeded
CheckRmse    Succeeded
Evaluate     Succeeded
Train        Succeeded
Preprocess   Succeeded


## 4. Metrics

The evaluate step wrote its report to S3. This is the same document the
gate reads.

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key="model/evaluation/evaluation.json")
report = json.loads(obj["Body"].read())

metrics = report["regression_metrics"]
rmse = metrics["rmse"]["value"]
baseline = metrics["baseline_rmse"]["value"]

print(f"rmse={rmse:.4f} r2={metrics['r2']['value']:.4f}")
print(f"baseline rmse={baseline:.4f}")

assert rmse < baseline, "model is no better than predicting the mean"

## 5. The gate

The run above passed, so a version was registered -- pending, not
approved.

In [ ]:
def list_versions():
    """Every version, newest first. Paginated: the API caps a page at 100."""
    pages = sm.get_paginator("list_model_packages").paginate(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        SortBy="CreationTime",
        SortOrder="Descending",
    )
    return [v for page in pages for v in page["ModelPackageSummaryList"]]


versions = list_versions()

for v in versions:
    print(f"v{v['ModelPackageVersion']:<3} {v['ModelApprovalStatus']:<22} {v['CreationTime']:%Y-%m-%d %H:%M}")

latest = versions[0]
assert latest["ModelApprovalStatus"] == "PendingManualApproval", latest["ModelApprovalStatus"]
print()
print("newest version is pending -- nothing deploys it until approved")

In [ ]:
strict = pipeline.start(parameters={"RmseThreshold": 50.0})
strict.wait()

print(strict.describe()["PipelineExecutionStatus"])

steps = {s["StepName"]: s["StepStatus"] for s in strict.list_steps()}
print(steps)

assert "Register" not in steps, "register ran despite failing the gate"

after = list_versions()
assert len(after) == len(versions), "a version was registered despite failing the gate"

print()
print(f"run succeeded, register skipped, still {len(after)} version(s)")

## 6. Approve

Approval is deliberately a separate act from training.

Studio: **Models > Model registry >** the group **>** the version **>
Update status**. Or from here:

In [ ]:
sm.update_model_package(
    ModelPackageArn=latest["ModelPackageArn"],
    ModelApprovalStatus="Approved",
)

print(f"v{latest['ModelPackageVersion']} approved")